# CLSM Super-Resolution (eSRRF) & Image Scanning Microscopy (ISM)

This notebook demonstrates the super-resolution reconstructions in `tttrlib`, all of which live on `tttrlib.CLSMSuperRes`:

1. **Photon-level eSRRF**: the Radial Gradient Convergence (RGC) prior map, and single-photon stochastic reassignment onto a finer raster.
2. **Physical ISM simulation**: a 2D tubulin microtubule phantom imaged through a 25-channel SPAD array.
3. **ISM reconstructions**: shift-vector estimation, Adaptive Pixel Reassignment (APR-ISM), and Focus-ISM background rejection.

The ISM algorithms follow the [BrightEyes-ISM](https://github.com/VicidominiLab/BrightEyes-ISM) reference implementation, and eSRRF follows [NanoJ-eSRRF](https://github.com/HenriquesLab/NanoJ-eSRRF).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
import tttrlib

# The tubulin phantom and the array-detector PSF model are simulation helpers
# shipped with the examples, not part of the library.
repo = Path.cwd().parent.parent
for extra in (repo / "examples" / "simulation", repo / "prototype" / "esrrf"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from generate_tubulin_phantom import generate_tubulin_phantom
from simulate import generate_ism_psf

print("1. Simulating Ground Truth Tubulin Network Phantom (25 nm/pixel)...")
ground_truth = generate_tubulin_phantom(n_filaments=12, size_px=128, pixel_size_nm=25.0)

print("2. Simulating Physical 5x5 SPAD Array Detector PSFs (0.5 AU pitch)...")
psf_sim = generate_ism_psf(
    na=1.4,
    wavelength_exc=488.0,
    wavelength_det=520.0,
    n_det=5,            # 5x5 SPAD detector array
    pitch_au=0.5,       # 0.5 Airy units pitch
    nx=64, ny=64,
    pixel_size_nm=25.0
)
channel_psfs = psf_sim['channel_psfs']         # Shape (25, 64, 64)
detector_offsets = psf_sim['detector_offsets'] # Shape (25, 2)

print("3. Convolving Phantom with SPAD Detector Array PSFs...")
n_det, ny, nx = 25, ground_truth.shape[0], ground_truth.shape[1]
spad_cube = np.zeros((n_det, ny, nx), dtype=np.float64)
for k in range(n_det):
    spad_cube[k] = fftconvolve(ground_truth, channel_psfs[k], mode='same')

# Add Poisson photon noise
rng = np.random.default_rng(1)
spad_cube = rng.poisson(np.clip(spad_cube * 150.0, 0, None)).astype(np.float64)
print(f"Simulated SPAD Array Cube Shape: {spad_cube.shape}")

In [ ]:
print("4. Performing ISM Super-Resolution Reconstructions from Simulation...")
# Standard CLSM Open-Pinhole Sum Image
clsm_sum = spad_cube.sum(axis=0)

# The shift vectors APR is built on: how far each detector element's image has
# to move to register onto the central element. They should trace out the
# detector lattice, scaled by roughly one half.
shifts = tttrlib.CLSMSuperRes.shift_vectors(spad_cube, usf=10)
print(f"Shift vectors (dy, dx), first three elements:\n{shifts[:3]}")

# Adaptive Pixel Reassignment (APR-ISM)
apr_ism = tttrlib.CLSMSuperRes.apr_reconstruction(spad_cube, usf=10)[0]

# Focus-ISM: in-focus signal, out-of-focus background, and the APR sum
focus_signal, focus_background, _ = tttrlib.CLSMSuperRes.focus_reconstruction(
    spad_cube, sigma_bound=2.0, calibration_size=16, parallelize=True
)

print("Reconstructions complete.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

panels = [
    ('A) Ground truth (tubulin phantom)', ground_truth),
    ('B) Confocal, open pinhole (channel sum)', clsm_sum),
    ('C) APR-ISM (adaptive pixel reassignment)', apr_ism),
    ('D) Focus-ISM: in-focus signal', focus_signal),
    ('E) Focus-ISM: out-of-focus background', focus_background),
    ('F) eSRRF map of the APR-ISM image',
     tttrlib.CLSMSuperRes.rgc_map(apr_ism, magnification=4, fwhm=2.5,
                                  sensitivity=1, intensity_weighting=True)),
]
for ax, (title, img) in zip(axes.ravel(), panels):
    ax.imshow(img, cmap='magma', origin='lower')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()